# Task 2: Architectural Variations, Hyperparameters & Augmentation
____________________________________________________________________________________________________

**Method: we change one thing at a time.** Every experiment starts from the baseline of Task 1 and changes only **one** setting. Everything else stays the same (same seed, same data split, same number of epochs), so the difference in the results comes from that setting only.

| group | what we try (baseline in **bold**) |
|---|---|
| depth (conv) | 2 / **3** / 5 conv layers |
| depth (fully connected) | 1 / **2** / 4 linear layers |
| kernel size | **3x3** / 5x5 / 7x7 |
| stride & padding | **max pool + same padding** / stride-2 conv / no padding |
| batch normalization | **off** / on |
| optimizer | SGD with momentum / **Adam** / AdamW |
| learning rate | 1e-2 / **1e-3** / 1e-4 |
| lr scheduler | **none** / StepLR / CosineAnnealing |
| data augmentation | **none** / basic / full / full + MixUp |

At the end we combine the best settings into a final model (`t2_best`).

**Note:** with `TRAIN = True`, an experiment is only trained if its results are not saved yet (useful if the training stops in the middle). With `TRAIN = False` everything is loaded from disk.

## 0. Google Colab setup

Only runs on Google Colab (skipped automatically when running locally):
1. mount Google Drive, so checkpoints and results are saved in the project folder on Drive
2. move into the `Notebooks/` folder so that `import utils` and the relative paths work

On Colab: **Runtime → Change runtime type → T4 GPU**. Change `PROJECT_DIR` if the project is in another folder of your Drive.

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DIR = "/content/drive/MyDrive/ai"  # folder of the project on Google Drive
    os.chdir(PROJECT_DIR + "/Notebooks")
    print("Working directory:", os.getcwd())

In [ ]:
%load_ext autoreload
%autoreload 2

import os

import torch
import torch.nn as nn
import torchvision
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import utils
from utils import classes

## 1. Settings

In [ ]:
TRAIN = True

num_epochs = 20   # same as Task 1
batch_size = 128
SEED = 42

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

criterion = nn.CrossEntropyLoss()

## 2. Data augmentation preview

- **basic**: `RandomCrop(32, padding=4)` + `RandomHorizontalFlip()`
- **full**: basic + `RandomAffine(degrees=15, translate=(0.1, 0.1))` + `ColorJitter(0.2, 0.2, 0.2)`
- **MixUp**: two images of the batch are blended, and the loss is blended with the same weight:

$$\tilde{x} = \lambda x_i + (1 - \lambda) x_j \qquad loss = \lambda \, CE(\hat{y}, y_i) + (1 - \lambda) \, CE(\hat{y}, y_j) \qquad \lambda \sim Beta(\alpha, \alpha)$$

Augmentation does not change the label of the image, it only shows the network more variations of the same object, which reduces overfitting.

In [ ]:
utils.set_seed(SEED)
aug_loader, _, _ = utils.get_dataloaders(batch_size=16, augmentation="full")
images, labels = next(iter(aug_loader))

print("full augmentation:")
utils.imshow(torchvision.utils.make_grid(images, nrow=16))
print(' '.join('%5s' % classes[labels[j]] for j in range(16)))

mixed_images, labels_a, labels_b, lam = utils.mixup_data(images, labels, alpha=0.2)
print(f"\nMixUp (lambda = {lam:.2f}):")
utils.imshow(torchvision.utils.make_grid(mixed_images, nrow=16))
print(' '.join('%5s' % classes[labels_a[j]] for j in range(16)))
print(' '.join('%5s' % classes[labels_b[j]] for j in range(16)))

## 3. List of experiments

Each experiment is a dictionary. Only the values that are different from the baseline are written, the others take the default values:

- `model_args`: arguments of `utils.CustomCNN` (default = same architecture as the baseline `Net`)
- `optimizer` = "adam", `lr` = 1e-3, `scheduler` = None, `augmentation` = "none", `mixup` = False

**Note:** SGD is used with lr = 1e-2 because 1e-3 is too small for SGD (we would only see a too slow learning rate). The learning rate experiments are done with Adam.

In [ ]:
experiments = [
    # baseline (same architecture and settings as Task 1)
    {"name": "t2_baseline", "group": "baseline"},

    # depth: number of conv layers and fully connected layers
    {"name": "t2_conv2", "group": "depth", "model_args": {"conv_channels": [32, 64]}},
    {"name": "t2_conv5", "group": "depth", "model_args": {"conv_channels": [32, 64, 128, 256, 256]}},
    {"name": "t2_fc1", "group": "depth", "model_args": {"fc_sizes": []}},             # Flatten -> Linear(10)
    {"name": "t2_fc4", "group": "depth", "model_args": {"fc_sizes": [512, 256, 128]}},

    # kernel size
    {"name": "t2_kernel5", "group": "kernel", "model_args": {"kernel_size": 5}},
    {"name": "t2_kernel7", "group": "kernel", "model_args": {"kernel_size": 7}},

    # stride and padding
    {"name": "t2_stride2", "group": "stride_padding", "model_args": {"use_stride": True}},   # stride-2 conv instead of max pool
    {"name": "t2_no_padding", "group": "stride_padding", "model_args": {"padding": "valid"}},

    # batch normalization
    {"name": "t2_batchnorm", "group": "batchnorm", "model_args": {"use_bn": True}},

    # optimizers
    {"name": "t2_sgd", "group": "optimizer", "optimizer": "sgd", "lr": 1e-2},
    {"name": "t2_adamw", "group": "optimizer", "optimizer": "adamw"},

    # initial learning rate (Adam)
    {"name": "t2_lr1e-2", "group": "learning_rate", "lr": 1e-2},
    {"name": "t2_lr1e-4", "group": "learning_rate", "lr": 1e-4},

    # learning rate schedulers
    {"name": "t2_step_lr", "group": "scheduler", "scheduler": "step"},
    {"name": "t2_cosine_lr", "group": "scheduler", "scheduler": "cosine"},

    # data augmentation
    {"name": "t2_aug_basic", "group": "augmentation", "augmentation": "basic"},
    {"name": "t2_aug_full", "group": "augmentation", "augmentation": "full"},
    {"name": "t2_aug_mixup", "group": "augmentation", "augmentation": "full", "mixup": True},
]
print(len(experiments), "experiments")

## 4. Function to run one experiment

1. create the DataLoaders (with the augmentation of the experiment)
2. create the model, optimizer and scheduler, then train (or load the saved results)
3. load the best checkpoint and evaluate it on the test set
4. return one row of the comparison table

In [ ]:
def run_experiment(exp):
    name = exp["name"]
    checkpoint_path = f"{utils.CHECKPOINT_DIR}/{name}.pth"
    history_path = f"{utils.RESULTS_DIR}/{name}.json"
    model_args = exp.get("model_args", {})
    epochs = exp.get("epochs", num_epochs)

    utils.set_seed(SEED)  # same initial weights and same batches order for every experiment
    train_loader, val_loader, test_loader = utils.get_dataloaders(batch_size=batch_size,
                                                                  augmentation=exp.get("augmentation", "none"))

    if TRAIN and not os.path.exists(history_path):
        print(f"\n===== Training {name} =====")
        model = utils.CustomCNN(**model_args).to(device)
        optimizer = utils.get_optimizer(exp.get("optimizer", "adam"), model.parameters(), exp.get("lr", 1e-3))
        scheduler = utils.get_scheduler(exp.get("scheduler"), optimizer, epochs)

        history = utils.train_model(model, train_loader, val_loader, criterion, optimizer, epochs, device,
                                    checkpoint_path, scheduler=scheduler, use_mixup=exp.get("mixup", False),
                                    checkpoint_info={"model_name": "CustomCNN", "model_args": model_args})
        utils.save_history(history, history_path)
    else:
        print(f"Loading {name}")
        history = utils.load_history(history_path)

    # best checkpoint -> test set
    model, _ = utils.load_checkpoint(checkpoint_path, device)
    test_loss, test_acc, _, _, _ = utils.evaluate(model, test_loader, criterion, device)

    best = history["best_epoch"] - 1
    result = {
        "name": name,
        "group": exp["group"],
        "parameters": utils.count_parameters(model),
        "receptive field": utils.receptive_field(model),
        "latency (ms)": utils.measure_latency(model, device),
        "peak val acc": max(history["val_acc"]),
        "generalization gap": history["train_acc"][best] - history["val_acc"][best],
        "best epoch": best + 1,
        "test acc": test_acc,
        "time/epoch (s)": np.mean(history["epoch_time"]),
    }
    return result, history

## 5. Run all experiments

About 19 runs x 20 epochs. Each run saves its best model in `Checkpoints/<name>.pth` and its history in `results/<name>.json`.

In [ ]:
results = []
histories = {}

for exp in experiments:
    result, history = run_experiment(exp)
    results.append(result)
    histories[exp["name"]] = history

## 6. Comparative benchmark table

- **parameters**: total number of weights of the model
- **latency**: inference time for one image
- **peak val acc**: best validation accuracy over all epochs
- **generalization gap**: train accuracy - validation accuracy at the best epoch (big gap = overfitting)

In [ ]:
results_df = pd.DataFrame(results).set_index("name")
results_df.to_csv(f"{utils.RESULTS_DIR}/task2_benchmark.csv")
results_df.round(4)

In [ ]:
# validation accuracy and generalization gap of every experiment
results_df[["peak val acc", "generalization gap"]].sort_values("peak val acc").plot.barh(figsize=(9, 8))
plt.title("Peak validation accuracy and generalization gap")
plt.grid(alpha=0.3)
utils.save_figure("task2_summary.png")
plt.show()

## 7. Validation curves for each group (compared to the baseline)

In [ ]:
groups = ["depth", "kernel", "stride_padding", "batchnorm", "optimizer", "learning_rate", "scheduler", "augmentation"]

for group in groups:
    names = ["t2_baseline"] + [exp["name"] for exp in experiments if exp["group"] == group]
    utils.plot_compare({name: histories[name] for name in names}, title=group, save_name=f"task2_{group}.png")

## 8. Receptive field

The receptive field is the size of the region of the input image that one value of the last feature map depends on. For each conv / pool layer:

$$rf_{l} = rf_{l-1} + (k_l - 1) \times j_{l-1} \qquad j_l = j_{l-1} \times s_l$$

where $k$ is the kernel size, $s$ the stride and $j$ the "jump" (distance in input pixels between two neighbouring values of the feature map).

More layers, bigger kernels and more downsampling give a bigger receptive field. When it covers the whole 32x32 image, the last layers can combine information from the full object.

In [ ]:
architecture_groups = ["baseline", "depth", "kernel", "stride_padding"]
rf_df = results_df[results_df["group"].isin(architecture_groups)]
rf_df[["parameters", "receptive field", "peak val acc", "generalization gap"]].sort_values("receptive field").round(4)

## 9. Final model: combination of the best settings

**To update after reading the table above:** keep the best option of each group.

The values below are a first guess (usually the best on CIFAR-10): wider conv layers + BatchNorm + dropout, AdamW with a cosine scheduler and full augmentation. We train it for more epochs because augmentation makes the training slower to converge.

In [ ]:
best_experiment = {
    "name": "t2_best",
    "group": "final",
    "model_args": {"conv_channels": [64, 128, 256], "use_bn": True, "dropout": 0.3},
    "optimizer": "adamw",
    "lr": 1e-3,
    "scheduler": "cosine",
    "augmentation": "full",
    "mixup": False,
    "epochs": 30,
}

best_result, best_history = run_experiment(best_experiment)
histories["t2_best"] = best_history
results_df = results_df.drop("t2_best", errors="ignore")  # in case the cell is run twice
results_df = pd.concat([results_df, pd.DataFrame([best_result]).set_index("name")])
results_df.to_csv(f"{utils.RESULTS_DIR}/task2_benchmark.csv")
results_df.loc[["t2_baseline", "t2_best"]].round(4)

In [ ]:
utils.plot_history(best_history, title="Best scratch CNN", save_name="task2_best_curves.png")

best_model, _ = utils.load_checkpoint(f"{utils.CHECKPOINT_DIR}/t2_best.pth", device)
_, _, test_loader = utils.get_dataloaders(batch_size=batch_size)
test_loss, test_acc, test_class_acc, test_preds, test_labels = utils.evaluate(best_model, test_loader, criterion, device)
print('Accuracy of the best model on the 10000 test images: %.2f %%' % (100 * test_acc))

utils.plot_confusion_matrix(test_labels, test_preds, title="Best scratch CNN - normalized confusion matrix (test)",
                            save_name="task2_best_confusion_matrix.png")
utils.plot_class_accuracy(test_class_acc, title="Best scratch CNN - per-class test accuracy",
                          save_name="task2_best_class_acc.png")

## Observations

*(to fill after training, with numbers from the table)*

- **Depth:**
- **Kernel size:**
- **Stride & padding:**
- **Batch normalization:**
- **Optimizer:**
- **Learning rate:**
- **Scheduler:**
- **Augmentation:** (expected: lower train accuracy, smaller generalization gap, needs more epochs)
- **Final model vs baseline:**